### In this notebook we are going to Clean data for our 5M (50,00,000) Row data, detection and removal of missing values, duplicate values and outliers if present. It also includes conversion of columns into their appropriate data type and reconstruction of target column

**Load the dataset from dataset directory**

In [4]:
import os
import pandas as pd
current_dir = os.getcwd()
relative_data_dir = os.path.abspath(os.path.join(current_dir, "..", "..", "dataset", "Dataset 3"))

dataset_candidate = os.path.join(relative_data_dir, "ecommerce_orders_dataset_50L.csv")

dataset_path = dataset_candidate if os.path.exists(dataset_candidate) else r"C:\Users\Acer\Desktop\E_Commerce_Return_Prediction\dataset\Dataset 3\ecommerce_orders_dataset_50L.csv"
df = pd.read_csv(dataset_path)
print(f"Primary Dataset loaded from: {dataset_path}")
print(df.head())
print(f"Dataset shape: {df.shape}")


Primary Dataset loaded from: C:\Users\Acer\Desktop\E_Commerce_Return_Prediction\dataset\Dataset 3\ecommerce_orders_dataset_50L.csv
        order_id   customer_id  order_date product_category     product_id  \
0  ORD0000000000  CUST06438651  2024-12-03          grocery  SKU-GRO-34044   
1  ORD0000000001  CUST04024143  2025-03-19             home  SKU-HOM-22793   
2  ORD0000000002  CUST08227616  2025-06-16          grocery  SKU-GRO-47426   
3  ORD0000000003  CUST05454291  2025-07-27      electronics  SKU-ELE-39207   
4  ORD0000000004  CUST04434141  2024-03-11             home  SKU-HOM-09643   

   quantity  unit_price  order_total payment_method  shipping_cost  \
0         5        4.83        31.59    credit_card           7.44   
1         1      101.25       101.25    credit_card           0.00   
2         1       13.88        18.06    credit_card           4.18   
3         4      416.63      1666.52    credit_card           0.00   
4         3      104.94       314.82         paypa

**1. Import libraries and inspect the dataset**

In [5]:
import pandas as pd

print("Dataset shape:", df.shape)
display(df.head())

print("\nData types:")
print(df.dtypes)

print("\nDataset information:")
df.info()

print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False))

print("\nDuplicate rows:", df.duplicated().sum())

Dataset shape: (5000000, 12)


,order_id,customer_id,order_date,product_category,product_id,quantity,unit_price,order_total,payment_method,shipping_cost,order_status,discount_applied
0,ORD0000000000,CUST06438651,2024-12-03,grocery,SKU-GRO-34044,5,4.83,31.59,credit_card,7.44,delivered,False
1,ORD0000000001,CUST04024143,2025-03-19,home,SKU-HOM-22793,1,101.25,101.25,credit_card,0.00,delivered,False
2,ORD0000000002,CUST08227616,2025-06-16,grocery,SKU-GRO-47426,1,13.88,18.06,credit_card,4.18,processing,False
3,ORD0000000003,CUST05454291,2025-07-27,electronics,SKU-ELE-39207,4,416.63,1666.52,credit_card,0.00,delivered,False
4,ORD0000000004,CUST04434141,2024-03-11,home,SKU-HOM-09643,3,104.94,314.82,paypal,0.00,delivered,False



Data types:
order_id             object
customer_id          object
order_date           object
product_category     object
product_id           object
quantity              int64
unit_price          float64
order_total         float64
payment_method       object
shipping_cost       float64
order_status         object
discount_applied       bool
dtype: object

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000000 entries, 0 to 4999999
Data columns (total 12 columns):
 #   Column            Dtype  
---  ------            -----  
 0   order_id          object 
 1   customer_id       object 
 2   order_date        object 
 3   product_category  object 
 4   product_id        object 
 5   quantity          int64  
 6   unit_price        float64
 7   order_total       float64
 8   payment_method    object 
 9   shipping_cost     float64
 10  order_status      object 
 11  discount_applied  bool   
dtypes: bool(1), float64(3), int64(1), object(7)
memory usage: 424.4

order_id            0
customer_id         0
order_date          0
product_category    0
product_id          0
quantity            0
unit_price          0
order_total         0
payment_method      0
shipping_cost       0
order_status        0
discount_applied    0
dtype: int64


Duplicate rows: 0


**As per output of code 1, there are no missing values in any of our columns so there's no need to remove missing values or have any need to remove duplicate rows**

**2. Validate values and identify invalid records**

In [6]:
numeric_columns = [
    "quantity",
    "unit_price",
    "order_total",
    "shipping_cost"
]

print("Negative values in numerical columns:")
for col in numeric_columns:
    print(col, (df[col] < 0).sum())

print("\nZero quantity:", (df["quantity"] == 0).sum())
print("Zero unit price:", (df["unit_price"] == 0).sum())

print("\nInvalid dates:", df["order_date"].isna().sum())

print("\nOrder status distribution:")
display(df["order_status"].value_counts(dropna=False))

print("\nCategorical values:")
for col in ["product_category", "payment_method"]:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False))

Negative values in numerical columns:
quantity 0
unit_price 0
order_total 0
shipping_cost 0

Zero quantity: 0
Zero unit price: 0

Invalid dates: 0

Order status distribution:


order_status
delivered     3547206
shipped        499914
returned       451819
processing     300401
cancelled      200660
Name: count, dtype: int64


Categorical values:

product_category
product_category
clothing       1100245
electronics     799007
home            700044
books           548789
beauty          500508
grocery         500254
sports          450780
toys            400373
Name: count, dtype: int64

payment_method
payment_method
credit_card    2200124
debit_card     1200815
paypal          899482
bnpl            399588
gift_card       299991
Name: count, dtype: int64


**3. Construct the return target (converting problem statement into binary class classification as we have 5 labels in order_status target column ['delivered' 'processing' 'shipped' 'cancelled' 'returned'] , here only delivered and returned holds importance rest are not required as we can't accurately determine wheter the product will be return just with the prediction of shipped and returned , cancelled label is removed completely as it has no role in further delivery process)**

In [7]:
# Inspect labels before filtering
print(df["order_status"].value_counts(dropna=False))

# Keep only completed outcome classes for this binary task
df = df[
    df["order_status"].isin(["delivered", "returned"])
].copy()

# Returned = 1, Delivered = 0
df["return_target"] = (
    df["order_status"] == "returned"
).astype("int8")

print("\nTarget distribution:")
display(df["return_target"].value_counts())

print("\nTarget percentages:")
display(
    (df["return_target"].value_counts(normalize=True) * 100)
    .rename("percentage")
)

order_status
delivered     3547206
shipped        499914
returned       451819
processing     300401
cancelled      200660
Name: count, dtype: int64

Target distribution:


return_target
0    3547206
1     451819
Name: count, dtype: int64


Target percentages:


return_target
0    88.701771
1    11.298229
Name: percentage, dtype: float64

**Encoding Our label of interest "returned" as 1 and delivered as "0" for better explainibilty**

**4. Converting order_date feature into datetime data type for easier date time parsing in future**

In [8]:
df["order_date"] = pd.to_datetime(
    df["order_date"],
    errors="coerce"
)

print(df["order_date"].dtype)
display(df[["order_date"]].head())

datetime64[ns]


,order_date
0,2024-12-03
1,2025-03-19
3,2025-07-27
4,2024-03-11
5,2025-02-02


**5. Detect outliers using the IQR method**

In [9]:
outlier_summary = []

for col in numeric_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_count = (
        (df[col] < lower_bound) |
        (df[col] > upper_bound)
    ).sum()

    outlier_summary.append({
        "feature": col,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": outlier_count,
        "outlier_percentage": outlier_count / len(df) * 100
    })

outlier_df = pd.DataFrame(outlier_summary)
display(outlier_df)

,feature,Q1,Q3,IQR,lower_bound,upper_bound,outlier_count,outlier_percentage
0,quantity,1.00,2.00,1.00,-0.500,3.500,479998,12.002876
1,unit_price,21.97,124.69,102.72,-132.110,278.770,404065,10.104088
2,order_total,41.07,218.42,177.35,-224.955,484.445,429747,10.746294
3,shipping_cost,0.00,5.74,5.74,-8.610,14.350,0,0.000000


**Should we drop outliers? We must check it by Comparing target-label percentages**

In [10]:
import pandas as pd

target_col = "order_status"  # Replace with your actual target column name
numeric_cols = ["quantity", "unit_price", "order_total"]

for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    outlier_mask = (
        (df[col] < q1 - 1.5 * iqr) |
        (df[col] > q3 + 1.5 * iqr)
    )

    print(f"\n{'=' * 50}")
    print(f"Feature: {col}")

    # Count records by outlier status and target category
    counts = pd.crosstab(
        outlier_mask.map({False: "Non-outlier", True: "Outlier"}),
        df[target_col]
    )

    print("\nCategory counts:")
    print(counts)

    # Percentage distribution within each group
    percentages = pd.crosstab(
        outlier_mask.map({False: "Non-outlier", True: "Outlier"}),
        df[target_col],
        normalize="index"
    ) * 100

    print("\nCategory percentages:")
    print(percentages.round(2))


Feature: quantity

Category counts:
order_status  delivered  returned
quantity                         
Non-outlier     3121330    397697
Outlier          425876     54122

Category percentages:
order_status  delivered  returned
quantity                         
Non-outlier       88.70     11.30
Outlier           88.72     11.28

Feature: unit_price

Category counts:
order_status  delivered  returned
unit_price                       
Non-outlier     3198448    396512
Outlier          348758     55307

Category percentages:
order_status  delivered  returned
unit_price                       
Non-outlier       88.97     11.03
Outlier           86.31     13.69

Feature: order_total

Category counts:
order_status  delivered  returned
order_total                      
Non-outlier     3173236    396042
Outlier          373970     55777

Category percentages:
order_status  delivered  returned
order_total                      
Non-outlier       88.90     11.10
Outlier           87.02     12.98

**6. inspect the extreme values**

In [11]:
cols = ["quantity", "unit_price", "order_total"]

for col in cols:
    print(f"\n{'=' * 50}")
    print(f"Feature: {col}")
    print(df[col].quantile(
        [0.90, 0.95, 0.99, 0.995, 0.999, 1.00]
    ))


Feature: quantity
0.900    4.0
0.950    5.0
0.990    6.0
0.995    6.0
0.999    6.0
1.000    6.0
Name: quantity, dtype: float64

Feature: unit_price
0.900     280.96000
0.950     433.41000
0.990     679.54000
0.995     754.13880
0.999     883.86904
1.000    1157.48000
Name: unit_price, dtype: float64

Feature: order_total
0.900     510.34000
0.950     803.56000
0.990    1790.19760
0.995    2281.07040
0.999    3390.17952
1.000    6504.90000
Name: order_total, dtype: float64


## Outlier Analysis and Treatment

### 1. Overview

Outlier detection was performed on the numerical features `quantity`, `unit_price`, `order_total`, and `shipping_cost` using the Interquartile Range (IQR) method. The analysis was conducted on approximately 5 million e-commerce records.

The IQR method identifies potential outliers using the following boundaries:

- **Lower Bound:** Q1 − 1.5 × IQR
- **Upper Bound:** Q3 + 1.5 × IQR

Observations outside these boundaries are flagged as potential outliers, but they are not necessarily invalid records.

### 2. Outlier Detection Results

| Feature | Outlier Count | Outlier Percentage | Maximum Value |
|---|---:|---:|---:|
| `quantity` | 600,193 | 12.00% | 6.00 |
| `unit_price` | 505,786 | 10.12% | 1,157.48 |
| `order_total` | 537,677 | 10.75% | 6,775.86 |
| `shipping_cost` | 0 | 0.00% | 9.99 |

### 3. Feature-Wise Analysis

**Quantity**

The median quantity is 1, and the maximum quantity is 6. Although approximately 12% of observations are flagged as outliers, the range is relatively small and may represent legitimate bulk purchases. The return rates are almost identical for outlier and non-outlier observations (11.28% and 11.30%, respectively).

**Decision:** Retain valid quantity values and investigate only records that violate business rules.

**Unit Price**

The median unit price is 51.76, while the 99th percentile is 679.54 and the maximum is 1,157.48. The mean (107.00) is substantially higher than the median, indicating a right-skewed distribution.

The return rate is 13.69% among outliers, compared with 11.03% among non-outliers. The return-rate visualization also suggests that the relationship between price and returns may be nonlinear.

**Decision:** Retain legitimate high-priced products. Investigate suspicious prices and consider a logarithmic transformation if appropriate for the selected machine learning model.

**Order Total**

The median order total is 82.92, the 99th percentile is approximately 1,790.20, and the maximum is 6,775.86. The distribution has a long upper tail, potentially reflecting legitimate high-value transactions.

The return rate among outliers is 12.98%, compared with 11.10% among non-outliers.

**Decision:** Retain valid order totals and investigate extreme transactions for consistency with quantity, unit price, shipping charges, discounts, and taxes, where applicable.

**Shipping Cost**

The median and first quartile are both zero, suggesting a zero-heavy distribution. No observations were flagged by the IQR method, but this does not guarantee that every shipping cost is valid.

**Decision:** Retain valid shipping costs and investigate unusual values using appropriate business rules.

### 4. Final Outlier Treatment Strategy

Based on the statistical analysis and observed return-rate patterns, automatic outlier removal is not recommended at this stage.

The following actions are proposed:

1. Retain legitimate extreme observations to preserve potentially useful information.
2. Investigate and correct genuinely invalid records according to business rules.
3. Consider logarithmic transformations for strongly right-skewed numerical features where appropriate.
4. Evaluate robust scaling if the selected model is sensitive to extreme values.
5. Compare model performance with and without optional transformations or outlier capping using a consistent validation strategy.
6. Fit preprocessing statistics using the training set only to prevent data leakage.

### 5. Conclusion

The analysis identified potential outliers in `quantity`, `unit_price`, and `order_total`, while no IQR outliers were detected in `shipping_cost`. The return-rate comparisons suggest that high unit prices and order totals may be associated with a higher proportion of returned orders, whereas quantity outliers show almost no difference.

Therefore, **the current decision is to retain valid outliers, investigate potentially invalid records, and evaluate alternative preprocessing techniques before making further changes**. This approach preserves useful transaction-level information while supporting reliable downstream machine learning analysis.

**Saving transformed dataset into resuable parquet file**

In [13]:
df.to_parquet("cleaned_ecommerce_orders.parquet", index=False)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
